# EfficientNet-B0 transfer learning - EuroSAT

EN3150 Assignment 03: an ImageNet-pretrained lightweight model for comparison with custom Model B and MobileNetV2.

This notebook currently establishes the shared dataset contract and builds an ImageNet-pretrained EfficientNet-B0 classifier. No training or test-performance evaluation is performed at this stage.

## Shared data and preprocessing

Use the exact committed `data/eurosat_split.csv` and local folders created by `Dataset.ipynb`: 18,900 training, 4,050 validation and 4,050 test images, in 10 classes. Run this notebook from the repository root after dataset generation. Never re-split, resize or augment these images.

**EfficientNet-specific input contract:** native 64x64 RGB JPEGs become float32 tensors in **0-255**, rather than Model B's 0-1 tensors. Keras EfficientNet-B0 includes its own rescaling and normalization; do not divide by 255 or apply additional ImageNet normalization here.

Reference: https://keras.io/api/applications/efficientnet/efficientnet_models/


In [12]:
import os
# Keep downloaded Keras weights in the workspace, outside version control.
os.environ["KERAS_HOME"] = os.path.abspath("data/keras_cache")
# Set before numerical-library imports. Restart the kernel if they are already loaded.
THREAD_ENV = ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS"]
for name in THREAD_ENV:
    os.environ[name] = "1"

import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = Path("data/eurosat")
SPLIT_CSV = Path("data/eurosat_split.csv")
IMG_SIZE = (64, 64)
SEED = 42
BATCH_SIZE = 4  # Conservative starting batch; verify memory before full training.
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_COUNTS = {"train": 18_900, "val": 4_050, "test": 4_050}

tf.keras.utils.set_random_seed(SEED)
print("Python:", platform.python_version(), "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none")
print("Thread settings:", {name: os.environ.get(name) for name in THREAD_ENV})
print("Class mapping:", dict(enumerate(CLASS_NAMES)))


Python: 3.12.2 | TensorFlow: 2.21.0
GPU: none
Thread settings: {'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'TF_NUM_INTRAOP_THREADS': '1', 'TF_NUM_INTEROP_THREADS': '1'}
Class mapping: {0: 'AnnualCrop', 1: 'Forest', 2: 'HerbaceousVegetation', 3: 'Highway', 4: 'Industrial', 5: 'Pasture', 6: 'PermanentCrop', 7: 'Residential', 8: 'River', 9: 'SeaLake'}


## 1. Validate the shared split and files

The checks below reject missing/duplicate manifest identities, unexpected class or split names, wrong split proportions and extra/missing local files. File identities are `(class, filename)` pairs. A SHA-256 identifies the exact manifest for later experiment records; it is not an image-content checksum.


In [13]:
if not SPLIT_CSV.is_file() or not DATA_DIR.is_dir():
    raise FileNotFoundError("Run Dataset.ipynb from the repository root to create the shared EuroSAT dataset first.")
manifest = pd.read_csv(SPLIT_CSV, dtype=str)
assert list(manifest.columns) == ["file", "label", "split"], "Unexpected manifest columns"
assert not manifest.isna().any().any(), "Missing manifest values"
assert not manifest.duplicated(["label", "file"]).any(), "An image appears in more than one manifest row"
assert set(manifest["label"]) == set(CLASS_NAMES), "Unexpected class labels"
assert set(manifest["split"]) == set(EXPECTED_COUNTS), "Unexpected split names"
assert manifest["file"].map(lambda name: Path(name).name == name and name.lower().endswith(".jpg")).all(), "Invalid filenames"
assert len(manifest) == 27_000, "Expected 27,000 images"

counts = pd.crosstab(manifest["label"], manifest["split"]).reindex(
    index=CLASS_NAMES, columns=list(EXPECTED_COUNTS),
)
assert not counts.isna().any().any(), "A class is absent from a split"
np.testing.assert_allclose(
    counts.to_numpy() / counts.sum(axis=1).to_numpy()[:, None],
    np.tile([0.70, 0.15, 0.15], (NUM_CLASSES, 1)), rtol=0, atol=1e-12,
)
for split, expected in EXPECTED_COUNTS.items():
    assert int(counts[split].sum()) == expected, f"Unexpected {split} count"
expected_paths = {f"{row.split}/{row.label}/{row.file}" for row in manifest.itertuples(index=False)}
actual_paths = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()}
missing, extra = sorted(expected_paths - actual_paths), sorted(actual_paths - expected_paths)
assert not missing and not extra, f"Local files differ: missing={missing[:5]}, extra={extra[:5]}"
SPLIT_SHA256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print(counts.to_string())
print("Split totals:", counts.sum().to_dict())
print("Manifest SHA-256:", SPLIT_SHA256)


split                 train  val  test
label                                 
AnnualCrop             2100  450   450
Forest                 2100  450   450
HerbaceousVegetation   2100  450   450
Highway                1750  375   375
Industrial             1750  375   375
Pasture                1400  300   300
PermanentCrop          1750  375   375
Residential            2100  450   450
River                  1750  375   375
SeaLake                2100  450   450
Split totals: {'train': 18900, 'val': 4050, 'test': 4050}
Manifest SHA-256: 0cf4cc1ca5601a80a4a9c7ce47fb3435fcf95a36eba01802e0d08baf719f0c3c


## 2. Decode without resizing or external normalisation

Decode with the original channel count and enforce 64x64x3, so malformed inputs fail instead of being silently resized or converted. Shuffle paths before decoding for training and preserve sorted validation/test order. Each factory call creates a fresh shuffle state for future reproducible experiments.

Use bounded parallelism and a one-batch prefetch to limit host memory. Do not cache the whole decoded dataset. Reproducible seeds do not guarantee identical floating-point results across hardware or TensorFlow versions.


In [14]:
def decode_efficientnet_image(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=0)
    image = tf.ensure_shape(image, IMG_SIZE + (3,))
    return tf.cast(image, tf.float32), label  # Deliberately keep raw 0-255 pixels.


split_paths, split_labels = {}, {}
for split in EXPECTED_COUNTS:
    rows = manifest.loc[manifest["split"] == split].sort_values(["label", "file"])
    split_paths[split] = [str(DATA_DIR / row.split / row.label / row.file) for row in rows.itertuples(index=False)]
    split_labels[split] = np.array([CLASS_NAMES.index(label) for label in rows["label"]], dtype=np.int32)


def make_efficientnet_dataset(split, batch_size=BATCH_SIZE, shuffle=None):
    """Create a new bounded-memory pipeline; shuffle only training by default."""
    if split not in EXPECTED_COUNTS:
        raise ValueError(f"Unknown split: {split}")
    if not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")
    if shuffle is None:
        shuffle = split == "train"
    if split != "train" and shuffle:
        raise ValueError("Validation and test order must remain fixed")
    ds = tf.data.Dataset.from_tensor_slices((split_paths[split], split_labels[split]))
    if shuffle:
        ds = ds.shuffle(EXPECTED_COUNTS[split], seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.deterministic = True
    options.threading.private_threadpool_size = 2
    ds = ds.with_options(options)
    return ds.map(decode_efficientnet_image, num_parallel_calls=2).batch(batch_size).prefetch(1)


## 3. Validate every decoded input

This is structural data validation, including test images, not model evaluation. Check actual decoded counts, shapes, finite pixels and valid labels. An explicit byte-for-byte comparison on a training image catches accidental 0-1 scaling, which a range check alone would miss.


In [15]:
for split, expected in EXPECTED_COUNTS.items():
    seen = 0
    # Disable training shuffle for this inspection, leaving training factories fresh.
    for images, labels in make_efficientnet_dataset(split, shuffle=False):
        tf.debugging.assert_type(images, tf.float32)
        tf.debugging.assert_equal(tf.shape(images)[1:], [64, 64, 3])
        tf.debugging.assert_all_finite(images, "Non-finite input pixels")
        tf.debugging.assert_greater_equal(images, 0.0)
        tf.debugging.assert_less_equal(images, 255.0)
        tf.debugging.assert_greater_equal(labels, 0)
        tf.debugging.assert_less(labels, NUM_CLASSES)
        seen += int(tf.shape(images)[0])
    assert seen == expected, f"Wrong decoded count for {split}"
    print(f"{split}: {seen:,} valid 64x64 RGB images")

sample_path = split_paths["train"][0]
original = tf.io.decode_jpeg(tf.io.read_file(sample_path), channels=0)
sample_batch, sample_labels = next(iter(make_efficientnet_dataset("train", shuffle=False)))
np.testing.assert_array_equal(sample_batch[0].numpy(), tf.cast(original, tf.float32).numpy())
assert float(tf.reduce_max(sample_batch[0])) > 1.0, "Raw pixel scale was lost"
assert int(sample_labels[0]) == int(split_labels["train"][0])
print("Training sample range:", float(tf.reduce_min(sample_batch[0])), "to", float(tf.reduce_max(sample_batch[0])))
print("Raw pixel values preserved exactly; no external normalization applied.")

train_ds = make_efficientnet_dataset("train")
val_ds = make_efficientnet_dataset("val")
test_ds = make_efficientnet_dataset("test")
print("Data pipeline ready. No model built or training started.")


train: 18,900 valid 64x64 RGB images
val: 4,050 valid 64x64 RGB images
test: 4,050 valid 64x64 RGB images
Training sample range: 81.0 to 206.0
Raw pixel values preserved exactly; no external normalization applied.
Data pipeline ready. No model built or training started.


## 4. ImageNet-pretrained backbone and EuroSAT head

Load EfficientNet-B0 with `include_top=False`, ImageNet weights and native 64x64 RGB inputs. Replace its ImageNet classifier with global average pooling, dropout 0.3 and a ten-class softmax layer. Retain the pretrained model's internal rescaling/normalization: inputs remain raw 0-255 float32.

Freeze the entire backbone for the initial head-training stage. Call it with `training=False` to keep BatchNorm moving statistics and internal stochastic layers in inference mode, including during subsequent fine-tuning. The head's dropout still follows the outer model's training mode. Later we will explicitly unfreeze selected convolutional blocks, keep BatchNorm frozen and recompile with a fresh low-learning-rate optimizer.

64x64 inputs satisfy this implementation's minimum size, but ImageNet weights were trained at a different resolution and on natural photographs. Transfer quality on satellite patches must be measured; it is not guaranteed by the pretrained model's ImageNet accuracy. The assignment's 100,000-trainable-parameter cap applies to custom Model B, not this pretrained baseline.

Weights are cached in ignored `data/keras_cache/models/`. The first run needs internet access. Use `weights="imagenet"` rather than silently falling back to random weights if downloading fails.


In [16]:
from tensorflow.keras import layers, models


def build_efficientnet_b0(num_classes=NUM_CLASSES, dropout_rate=0.3):
    """Build a frozen ImageNet backbone with a freshly initialized EuroSAT head."""
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,),
    )
    backbone.trainable = False
    inputs = layers.Input(shape=IMG_SIZE + (3,), name="rgb_pixels_0_255")
    features = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(features)
    x = layers.Dropout(dropout_rate, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="land_use_probabilities")(x)
    model = models.Model(inputs, outputs, name="EuroSAT_EfficientNetB0")
    return model, backbone


tf.keras.utils.set_random_seed(SEED)
efficientnet_model, efficientnet_backbone = build_efficientnet_b0()
efficientnet_model.summary()


Model: "EuroSAT_EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rgb_pixels_0_255 (InputLayer)   │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 2, 2, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ land_use_probabilities (Dense)  │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [17]:
n_trainable = int(sum(np.prod(weight.shape) for weight in efficientnet_model.trainable_weights))
n_non_trainable = int(sum(np.prod(weight.shape) for weight in efficientnet_model.non_trainable_weights))
n_total = int(efficientnet_model.count_params())
assert n_total == n_trainable + n_non_trainable
assert not efficientnet_backbone.trainable and not efficientnet_backbone.trainable_weights
assert efficientnet_model.input_shape == (None, 64, 64, 3)
assert efficientnet_model.output_shape == (None, NUM_CLASSES)
head = efficientnet_model.get_layer("land_use_probabilities")
expected_head_params = int(efficientnet_backbone.output_shape[-1]) * NUM_CLASSES + NUM_CLASSES
assert n_trainable == head.count_params() == expected_head_params
assert any(isinstance(layer, layers.Rescaling) for layer in efficientnet_backbone.layers)

# Interface validation only; no accuracy measurement or test prediction.
x_check, _ = next(iter(make_efficientnet_dataset("val", batch_size=4)))
probabilities = efficientnet_model(x_check, training=False).numpy()
assert probabilities.shape == (len(x_check), NUM_CLASSES)
assert np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-5)

# The frozen backbone must not update BatchNorm statistics in outer training mode.
batchnorm = next(layer for layer in efficientnet_backbone.layers if isinstance(layer, layers.BatchNormalization))
mean_before, variance_before = batchnorm.moving_mean.numpy().copy(), batchnorm.moving_variance.numpy().copy()
_ = efficientnet_model(x_check[:1], training=True)
np.testing.assert_array_equal(batchnorm.moving_mean.numpy(), mean_before)
np.testing.assert_array_equal(batchnorm.moving_variance.numpy(), variance_before)
print(f"Total parameters: {n_total:,}")
print(f"Trainable parameters (head only): {n_trainable:,}")
print(f"Non-trainable parameters: {n_non_trainable:,}")
print("Backbone feature-map shape:", efficientnet_backbone.output_shape)
print("Classifier output shape:", probabilities.shape)
print("Output probabilities and frozen BatchNorm checks passed. No training performed.")


Total parameters: 4,062,381
Trainable parameters (head only): 12,810
Non-trainable parameters: 4,049,571
Backbone feature-map shape: (None, 2, 2, 1280)
Classifier output shape: (4, 10)
Output probabilities and frozen BatchNorm checks passed. No training performed.


In [18]:
# Official Keras no-top B0 weights; Keras also checks this file's published hash on download.
WEIGHTS_FILE = Path(os.environ["KERAS_HOME"]) / "models" / "efficientnetb0_notop.h5"
EXPECTED_WEIGHTS_MD5 = "50bc09e76180e00e4465e1a485ddc09d"
assert WEIGHTS_FILE.is_file(), "Expected pretrained weight cache was not found"
weight_bytes = WEIGHTS_FILE.read_bytes()
assert hashlib.md5(weight_bytes).hexdigest() == EXPECTED_WEIGHTS_MD5, "Pretrained weight checksum differs"
PRETRAINED_PROVENANCE = {
    "architecture": "EfficientNetB0", "pretrained_dataset": "ImageNet", "include_top": False,
    "input_shape": [64, 64, 3], "input_range": [0, 255], "head_dropout": 0.3,
    "weights_url": "https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5",
    "weights_md5": EXPECTED_WEIGHTS_MD5, "weights_sha256": hashlib.sha256(weight_bytes).hexdigest(),
    "tensorflow": tf.__version__, "keras": tf.keras.__version__,
    "python": platform.python_version(), "split_sha256": SPLIT_SHA256,
}
del weight_bytes
print(PRETRAINED_PROVENANCE)


{'architecture': 'EfficientNetB0', 'pretrained_dataset': 'ImageNet', 'include_top': False, 'input_shape': [64, 64, 3], 'input_range': [0, 255], 'head_dropout': 0.3, 'weights_url': 'https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5', 'weights_md5': '50bc09e76180e00e4465e1a485ddc09d', 'weights_sha256': '0683edcf6151811fb3ddd6f098ac48df644d74592c4919a5acdee1c37554add2', 'tensorflow': '2.21.0', 'keras': '3.15.1', 'python': '3.12.2', 'split_sha256': '0cf4cc1ca5601a80a4a9c7ce47fb3435fcf95a36eba01802e0d08baf719f0c3c'}


## Next stage

Implement frozen-head training and upper-block fine-tuning with checkpointing and saved experiment metadata. Recompile after changing trainability. Use training/validation data for selection and keep test evaluation disabled until the recipe is frozen. No trained-model export or final performance claim is produced in this step.
